# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.")

for rset in record_sets:
    print(f"Record set name: {rset.name} (@id: {rset.id})")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}) | Data type: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> **Tip:** Choose a key record set for your analysis if there are multiple.

In [ ]:
# Extract data from each record set into DataFrames
record_set_ids = [rset.id for rset in record_sets]

dataframes = {}
for rid in record_set_ids:
    rows = list(dataset.records(record_set=rid))
    df = pd.DataFrame(rows)
    dataframes[rid] = df
    print(f"Loaded {len(df)} records for record set {rid}")

# As an example, take the first record set for demonstration
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field and a group-by field for demonstration
# Replace these with actual @id values from the output above as needed.
main_df = dataframes[main_record_set_id]

# Try to select one numeric field and one categorical field
numeric_field_candidates = [col for col in main_df.columns if main_df[col].dtype in ['float64', 'int64']]
if len(numeric_field_candidates) == 0:
    # Try to convert candidate columns to numeric and use the first successful one
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            if main_df[col].dtype in ['float64', 'int64']:
                numeric_field_candidates.append(col)
        except:
            continue

if len(numeric_field_candidates) > 0:
    numeric_field = numeric_field_candidates[0]
    print(f"Using numeric field: {numeric_field}")

    threshold = main_df[numeric_field].mean()
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f})")
    display(filtered_df.head())

    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_col]].head())
    
    # Try grouping by a candidate field with few unique values
    group_field = None
    for col in main_df.columns:
        if main_df[col].dtype == 'object' and main_df[col].nunique() < len(main_df)//3 and col != numeric_field:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for demonstration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_field_candidates) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the FAIR² dataset describing protein abundance and modification data from mass spectrometry analysis of extracellular vesicles. Using `mlcroissant`, we reviewed the record structure, loaded data, filtered and normalized numeric fields, and visualized data distributions. Customize the code blocks and fields further to suit your research questions and analysis tasks.